In [1]:
pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install vectorbt

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install TA-lib

Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install --upgrade statsmodels

In [5]:
!pip install fredapi pandas

In [6]:
import scipy
import statsmodels.api as sm

In [7]:
import yfinance as yf
import numpy as np
import pandas as pd
import talib
from datetime import datetime

In [74]:
# import data from ticker 

ydf = yf.download("NVDA",
                  start="2015-01-01",
                  end="2026-07-05",
                  progress=False)

In [75]:
print(f"Downloaded {len(ydf)} rows of data.")

Downloaded 2891 rows of data.


In [76]:
ydf

Price,Close,High,Low,Open,Volume
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA
Date,,,,,
2015-01-02,0.482985,0.486584,0.475307,0.482985,113680000
2015-01-05,0.474827,0.484425,0.472668,0.482985,197952000
2015-01-06,0.460432,0.476027,0.459952,0.475547,197764000
2015-01-07,0.459232,0.467870,0.457792,0.463791,321808000
2015-01-08,0.476507,0.479386,0.464270,0.464510,283780000
...,...,...,...,...,...
2026-06-26,192.529999,195.550003,191.220001,193.119995,179304100
2026-06-29,194.970001,196.179993,189.800003,193.850006,148835700


In [77]:
ydf.columns

MultiIndex([( 'Close', 'NVDA'),
            (  'High', 'NVDA'),
            (   'Low', 'NVDA'),
            (  'Open', 'NVDA'),
            ('Volume', 'NVDA')],
           names=['Price', 'Ticker'])

In [78]:
# Create column for the length of the candle body and daily price action (Total candle range)
ydf['Candle_Body'] = (ydf[('Close', 'NVDA')] - ydf[('Open', 'NVDA')]).abs()
ydf['Total price action'] = (ydf[('High', 'NVDA')] - ydf[('Low', 'NVDA')])

# Create a column to determine if Candle is bullish, bearish or a doji
doji = abs(ydf['Close', 'NVDA'] - ydf['Open', 'NVDA']) <= (0.10 * (ydf['High', 'NVDA'] - ydf['Close', 'NVDA']))
ydf['Bullish/Bearish'] = np.where(
    doji,
    'Doji',
    np.where((ydf['Close', 'NVDA'] > ydf['Open', 'NVDA']), 'Bullish', 'Bearish')
)

# Create column for candle body percentage by dividing the candle body by the total price action of the day
ydf['Body %'] = (ydf['Candle_Body'] / ydf['Total price action']) * 100

# Change total price action column to Candle_range 

ydf = ydf.rename(columns={'Total price action': 'Candle_Range'})

ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,
Date,,,,,,,,,
2015-01-02,0.482985,0.486584,0.475307,0.482985,113680000,0.000000,0.011277,Doji,0.000000
2015-01-05,0.474827,0.484425,0.472668,0.482985,197952000,0.008158,0.011757,Bearish,69.387800
2015-01-06,0.460432,0.476027,0.459952,0.475547,197764000,0.015116,0.016075,Bearish,94.029920
2015-01-07,0.459232,0.467870,0.457792,0.463791,321808000,0.004559,0.010077,Bearish,45.237879
2015-01-08,0.476507,0.479386,0.464270,0.464510,283780000,0.011997,0.015116,Bullish,79.364899
...,...,...,...,...,...,...,...,...,...
2026-06-26,192.529999,195.550003,191.220001,193.119995,179304100,0.589996,4.330002,Bearish,13.625776
2026-06-29,194.970001,196.179993,189.800003,193.850006,148835700,1.119995,6.379990,Bullish,17.554811


In [79]:
ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,
Date,,,,,,,,,
2015-01-02,0.482985,0.486584,0.475307,0.482985,113680000,0.000000,0.011277,Doji,0.000000
2015-01-05,0.474827,0.484425,0.472668,0.482985,197952000,0.008158,0.011757,Bearish,69.387800
2015-01-06,0.460432,0.476027,0.459952,0.475547,197764000,0.015116,0.016075,Bearish,94.029920
2015-01-07,0.459232,0.467870,0.457792,0.463791,321808000,0.004559,0.010077,Bearish,45.237879
2015-01-08,0.476507,0.479386,0.464270,0.464510,283780000,0.011997,0.015116,Bullish,79.364899
...,...,...,...,...,...,...,...,...,...
2026-06-26,192.529999,195.550003,191.220001,193.119995,179304100,0.589996,4.330002,Bearish,13.625776
2026-06-29,194.970001,196.179993,189.800003,193.850006,148835700,1.119995,6.379990,Bullish,17.554811


In [187]:
#Add 5, 10, 20, 50, and 200 day EMA Columns 

ydf['5_EMA'] = ydf['Close', 'NVDA'].ewm(span=5, adjust=False).mean()
ydf['10_EMA'] = ydf['Close', 'NVDA'].ewm(span=10, adjust=False).mean()
ydf['20_EMA'] = ydf['Close', 'NVDA'].ewm(span=20, adjust=False).mean()
ydf['50_EMA'] = ydf['Close', 'NVDA'].ewm(span=50, adjust=False).mean()
ydf['200_EMA'] = ydf['Close', 'NVDA'].ewm(span=200, adjust=False).mean()

ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,10_EMA,20_EMA,50_EMA,200_EMA,RSI,Upper_wick,Lower_wick,Upper_wick_percentage,Lower_wick_percentage,Wick_to_body_ratio,gap_size,gap_percentage,5EMA_touched,10EMA_touched,20EMA_touched,50EMA_touched,200EMA_touched,structure,confirmed_trend,EMA_Bullish,EMA_Alignment,5_day_return
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,,,,,,,,,,,,,,,,,,,,,,
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2015-01-02,0.482985,0.486584,0.475307,0.482985,113680000,0.000000,0.011277,Doji,0.000000,0.482985,0.482985,0.482985,0.482985,0.482985,NaN,0.003599,0.007678,31.915002,68.084998,inf,NaN,NaN,True,True,True,True,True,HL,0,False,divergence,0.009439
2015-01-05,0.474827,0.484425,0.472668,0.482985,197952000,0.008158,0.011757,Bearish,69.387800,0.480266,0.481502,0.482208,0.482665,0.482904,NaN,0.001440,0.002159,12.244977,18.367223,0.441176,-5.664483e-08,-0.000012,True,True,True,True,True,HL,0,False,Bullish,0.005053
2015-01-06,0.460432,0.476027,0.459952,0.475547,197764000,0.015116,0.016075,Bearish,94.029920,0.473655,0.477671,0.480134,0.481793,0.482681,NaN,0.000480,0.000480,2.985040,2.985040,0.063491,7.198833e-04,0.151609,True,False,False,False,False,HL,0,False,Bullish,-0.024492
2015-01-07,0.459232,0.467870,0.457792,0.463791,321808000,0.004559,0.010077,Bearish,45.237879,0.468847,0.474318,0.478144,0.480909,0.482447,NaN,0.004079,0.001440,40.476326,14.285795,1.210537,3.359050e-03,0.729544,False,False,False,False,False,LH,0,False,Bullish,-0.031348
2015-01-08,0.476507,0.479386,0.464270,0.464510,283780000,0.011997,0.015116,Bullish,79.364899,0.471400,0.474716,0.477988,0.480736,0.482388,NaN,0.002879,0.000240,19.047727,1.587374,0.260003,5.278483e-03,1.149415,True,True,True,False,True,HH,-1,False,Bullish,0.013091
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-26,192.529999,195.550003,191.220001,193.119995,179304100,0.589996,4.330002,Bearish,13.625776,198.058194,201.816238,205.671428,205.461208,188.614068,37.494618,2.430008,1.309998,56.120252,30.253972,6.339032,-2.620010e+00,-1.338516,False,False,False,False,False,LH,0,False,Bearish,NaN
2026-06-29,194.970001,196.179993,189.800003,193.850006,148835700,1.119995,6.379990,Bullish,17.554811,197.028796,200.571468,204.652244,205.049789,188.677311,40.084358,1.209991,4.050003,18.965414,63.479775,4.696444,1.320007e+00,0.685611,False,False,False,False,False,HH,-1,False,Bullish,NaN


In [81]:
import talib

In [82]:
ydf['RSI'] = talib.RSI(ydf['Close', 'NVDA'])
ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,10_EMA,20_EMA,50_EMA,200_EMA,RSI
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,,,,,
Date,,,,,,,,,,,,,,,
2015-01-02,0.482985,0.486584,0.475307,0.482985,113680000,0.000000,0.011277,Doji,0.000000,0.482985,0.482985,0.482985,0.482985,0.482985,NaN
2015-01-05,0.474827,0.484425,0.472668,0.482985,197952000,0.008158,0.011757,Bearish,69.387800,0.480266,0.481502,0.482208,0.482665,0.482208,NaN
2015-01-06,0.460432,0.476027,0.459952,0.475547,197764000,0.015116,0.016075,Bearish,94.029920,0.473655,0.477671,0.480134,0.481793,0.480134,NaN
2015-01-07,0.459232,0.467870,0.457792,0.463791,321808000,0.004559,0.010077,Bearish,45.237879,0.468847,0.474318,0.478144,0.480909,0.478144,NaN
2015-01-08,0.476507,0.479386,0.464270,0.464510,283780000,0.011997,0.015116,Bullish,79.364899,0.471400,0.474716,0.477988,0.480736,0.477988,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-26,192.529999,195.550003,191.220001,193.119995,179304100,0.589996,4.330002,Bearish,13.625776,198.058194,201.816238,205.671428,205.461208,205.671428,37.494618
2026-06-29,194.970001,196.179993,189.800003,193.850006,148835700,1.119995,6.379990,Bullish,17.554811,197.028796,200.571468,204.652244,205.049789,204.652244,40.084358


In [83]:
import vectorbt as vbt

In [106]:
#Calculate the upper wick by subtracking the high from either the open or close. Use max function and axis=1 to choose the option closest to the upperwick
ydf['Upper_wick'] = abs(ydf['High', 'NVDA'] - ydf[[('Open', 'NVDA'), ('Close', 'NVDA')]].max(axis=1))
# Caculate the same for the lower wick using the min function with axis=1 so it chooses the value closest to the lower wick between the open and close
ydf['Lower_wick'] = abs(ydf['Low', 'NVDA'] - ydf[[('Open', 'NVDA'), ('Close', 'NVDA')]].min(axis=1))

# Calculate wick percentage by dividing the absolute value of the upper or lower wick by the candle ranage (total candle length)
ydf['Upper_wick_percentage'] = abs(ydf['Upper_wick'] / ydf['Candle_Range']) * 100
ydf['Lower_wick_percentage'] = abs(ydf['Lower_wick'] / ydf['Candle_Range']) * 100

# Calculate Wick to body ratio by dividing the sum of the wicks by the candle body
ydf['Wick_to_body_ratio'] = (ydf['Lower_wick'] + ydf['Upper_wick']) / ydf['Candle_Body']

# Calsulate the gap size 
ydf['gap_size'] = ydf['Open', 'NVDA'] - ydf['Close', 'NVDA'].shift(1)
# Calculate gap percentage 
ydf['gap_percentage'] = (ydf['gap_size'] / ydf['Close', 'NVDA'].shift(1)) * 100

# Caluclate distance from EMA distnace Percentage 
ydf['5EMA_touched'] = np.where((ydf['5_EMA'] >= ydf['Low', 'NVDA']) & (ydf['5_EMA'] <= ydf['High', 'NVDA']), 'True', 'False')
ydf['10EMA_touched'] = np.where((ydf['10_EMA'] >= ydf['Low', 'NVDA']) & (ydf['10_EMA'] <= ydf['High', 'NVDA']), 'True', 'False')
ydf['20EMA_touched'] = np.where((ydf['20_EMA'] >= ydf['Low', 'NVDA']) & (ydf['20_EMA'] <= ydf['High', 'NVDA']), 'True', 'False')
ydf['50EMA_touched'] = np.where((ydf['50_EMA'] >= ydf['Low', 'NVDA']) & (ydf['50_EMA'] <= ydf['High', 'NVDA']), 'True', 'False')
ydf['200EMA_touched'] = np.where((ydf['200_EMA'] >= ydf['Low', 'NVDA']) & (ydf['200_EMA'] <= ydf['High', 'NVDA']), 'True', 'False')
ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,...,Wick_to_body_ratio,gap_size,gap_percentage,5EMA_touched,10EMA_touched,20EMA_touched,50EMA_touched,200EMA_touched,structure,confirmed_trend
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,...,,,,,,,,,,
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,0.482985,0.486584,0.475307,0.482985,113680000,0.000000,0.011277,Doji,0.000000,0.482985,...,inf,NaN,NaN,True,True,True,True,True,HL,0
2015-01-05,0.474827,0.484425,0.472668,0.482985,197952000,0.008158,0.011757,Bearish,69.387800,0.480266,...,0.441176,-5.664483e-08,-0.000012,True,True,True,True,True,HL,0
2015-01-06,0.460432,0.476027,0.459952,0.475547,197764000,0.015116,0.016075,Bearish,94.029920,0.473655,...,0.063491,7.198833e-04,0.151609,True,False,False,False,False,HL,0
2015-01-07,0.459232,0.467870,0.457792,0.463791,321808000,0.004559,0.010077,Bearish,45.237879,0.468847,...,1.210537,3.359050e-03,0.729544,False,False,False,False,False,LH,0
2015-01-08,0.476507,0.479386,0.464270,0.464510,283780000,0.011997,0.015116,Bullish,79.364899,0.471400,...,0.260003,5.278483e-03,1.149415,True,True,True,False,True,HH,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-26,192.529999,195.550003,191.220001,193.119995,179304100,0.589996,4.330002,Bearish,13.625776,198.058194,...,6.339032,-2.620010e+00,-1.338516,False,False,False,False,False,LH,0
2026-06-29,194.970001,196.179993,189.800003,193.850006,148835700,1.119995,6.379990,Bullish,17.554811,197.028796,...,4.696444,1.320007e+00,0.685611,False,False,False,False,False,HH,0


**Now let's create some features to help us determine market trend.**|

In [137]:
pd.set_option('display.max_columns', None)

In [190]:
# We'll start by determining higher highs, higher lows, lower highs, and lower lows 

h_high = ydf['Close', 'NVDA'] > ydf['Close', 'NVDA'].shift(1)
h_low = ydf['Close', 'NVDA'] > ydf['Close', 'NVDA'].shift(-1)
l_high = ydf['Close', 'NVDA'] < ydf['Close', 'NVDA'].shift(1)
l_low = ydf['Close', 'NVDA'] < ydf['Close', 'NVDA'].shift(-1)

# Now let's set parameters on how we want to search for higher highs, higher lows, lower highs and lower lows 

HH = h_high.rolling(5).sum()
HL = h_low.rolling(5).sum()
LH = l_high.rolling(5).sum()
LL = l_low.rolling(5).sum()

# Now let's determine the rules for our uptrend and downtrend 

Uptrend = (HH >= 2) & (HL >= 2)
Downtrend = (LH >= 2) & (LL >= 2) 

#Now let's bring it all together to determine uptrend 

ydf['confirmed_trend'] = np.select(
    [Uptrend, Downtrend], [1, -1],
    default=0
)
(ydf['confirmed_trend'] == 1).sum()
ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,10_EMA,20_EMA,50_EMA,200_EMA,RSI,Upper_wick,Lower_wick,Upper_wick_percentage,Lower_wick_percentage,Wick_to_body_ratio,gap_size,gap_percentage,5EMA_touched,10EMA_touched,20EMA_touched,50EMA_touched,200EMA_touched,structure,confirmed_trend,EMA_Bullish,EMA_Alignment,5_day_return
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,,,,,,,,,,,,,,,,,,,,,,
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2015-01-02,0.482985,0.486584,0.475307,0.482985,113680000,0.000000,0.011277,Doji,0.000000,0.482985,0.482985,0.482985,0.482985,0.482985,NaN,0.003599,0.007678,31.915002,68.084998,inf,NaN,NaN,True,True,True,True,True,HL,0,False,divergence,0.009439
2015-01-05,0.474827,0.484425,0.472668,0.482985,197952000,0.008158,0.011757,Bearish,69.387800,0.480266,0.481502,0.482208,0.482665,0.482904,NaN,0.001440,0.002159,12.244977,18.367223,0.441176,-5.664483e-08,-0.000012,True,True,True,True,True,HL,0,False,Bearish,0.005053
2015-01-06,0.460432,0.476027,0.459952,0.475547,197764000,0.015116,0.016075,Bearish,94.029920,0.473655,0.477671,0.480134,0.481793,0.482681,NaN,0.000480,0.000480,2.985040,2.985040,0.063491,7.198833e-04,0.151609,True,False,False,False,False,HL,0,False,Bearish,-0.024492
2015-01-07,0.459232,0.467870,0.457792,0.463791,321808000,0.004559,0.010077,Bearish,45.237879,0.468847,0.474318,0.478144,0.480909,0.482447,NaN,0.004079,0.001440,40.476326,14.285795,1.210537,3.359050e-03,0.729544,False,False,False,False,False,LH,0,False,Bearish,-0.031348
2015-01-08,0.476507,0.479386,0.464270,0.464510,283780000,0.011997,0.015116,Bullish,79.364899,0.471400,0.474716,0.477988,0.480736,0.482388,NaN,0.002879,0.000240,19.047727,1.587374,0.260003,5.278483e-03,1.149415,True,True,True,False,True,HH,-1,False,Bearish,0.013091
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-26,192.529999,195.550003,191.220001,193.119995,179304100,0.589996,4.330002,Bearish,13.625776,198.058194,201.816238,205.671428,205.461208,188.614068,37.494618,2.430008,1.309998,56.120252,30.253972,6.339032,-2.620010e+00,-1.338516,False,False,False,False,False,LH,0,False,divergence,NaN
2026-06-29,194.970001,196.179993,189.800003,193.850006,148835700,1.119995,6.379990,Bullish,17.554811,197.028796,200.571468,204.652244,205.049789,188.677311,40.084358,1.209991,4.050003,18.965414,63.479775,4.696444,1.320007e+00,0.685611,False,False,False,False,False,HH,-1,False,divergence,NaN


In [191]:
ydf.drop(columns=[('EMA_Bullish', "")])

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,10_EMA,20_EMA,50_EMA,200_EMA,RSI,Upper_wick,Lower_wick,Upper_wick_percentage,Lower_wick_percentage,Wick_to_body_ratio,gap_size,gap_percentage,5EMA_touched,10EMA_touched,20EMA_touched,50EMA_touched,200EMA_touched,structure,confirmed_trend,EMA_Alignment,5_day_return
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,,,,,,,,,,,,,,,,,,,,,
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2015-01-02,0.482985,0.486584,0.475307,0.482985,113680000,0.000000,0.011277,Doji,0.000000,0.482985,0.482985,0.482985,0.482985,0.482985,NaN,0.003599,0.007678,31.915002,68.084998,inf,NaN,NaN,True,True,True,True,True,HL,0,divergence,0.009439
2015-01-05,0.474827,0.484425,0.472668,0.482985,197952000,0.008158,0.011757,Bearish,69.387800,0.480266,0.481502,0.482208,0.482665,0.482904,NaN,0.001440,0.002159,12.244977,18.367223,0.441176,-5.664483e-08,-0.000012,True,True,True,True,True,HL,0,Bearish,0.005053
2015-01-06,0.460432,0.476027,0.459952,0.475547,197764000,0.015116,0.016075,Bearish,94.029920,0.473655,0.477671,0.480134,0.481793,0.482681,NaN,0.000480,0.000480,2.985040,2.985040,0.063491,7.198833e-04,0.151609,True,False,False,False,False,HL,0,Bearish,-0.024492
2015-01-07,0.459232,0.467870,0.457792,0.463791,321808000,0.004559,0.010077,Bearish,45.237879,0.468847,0.474318,0.478144,0.480909,0.482447,NaN,0.004079,0.001440,40.476326,14.285795,1.210537,3.359050e-03,0.729544,False,False,False,False,False,LH,0,Bearish,-0.031348
2015-01-08,0.476507,0.479386,0.464270,0.464510,283780000,0.011997,0.015116,Bullish,79.364899,0.471400,0.474716,0.477988,0.480736,0.482388,NaN,0.002879,0.000240,19.047727,1.587374,0.260003,5.278483e-03,1.149415,True,True,True,False,True,HH,-1,Bearish,0.013091
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-26,192.529999,195.550003,191.220001,193.119995,179304100,0.589996,4.330002,Bearish,13.625776,198.058194,201.816238,205.671428,205.461208,188.614068,37.494618,2.430008,1.309998,56.120252,30.253972,6.339032,-2.620010e+00,-1.338516,False,False,False,False,False,LH,0,divergence,NaN
2026-06-29,194.970001,196.179993,189.800003,193.850006,148835700,1.119995,6.379990,Bullish,17.554811,197.028796,200.571468,204.652244,205.049789,188.677311,40.084358,1.209991,4.050003,18.965414,63.479775,4.696444,1.320007e+00,0.685611,False,False,False,False,False,HH,-1,divergence,NaN


In [188]:
# Now Let's work with out EMA's and see if we can determine a stock's direction based on how they stack out

EMA_Bullish = (
   (ydf['5_EMA', ""] > ydf['10_EMA', ""]) &
   (ydf['10_EMA', ""] > ydf['20_EMA', ""]) &
   (ydf['20_EMA', ""] > ydf['50_EMA', ""]) & 
    (ydf['50_EMA', ""] > ydf['200_EMA', ""])
)

EMA_Bearish = (
    (ydf['5_EMA', ""] < ydf['10_EMA', ""]) &
   (ydf['10_EMA', ""] < ydf['20_EMA', ""]) &
   (ydf['20_EMA', ""] < ydf['50_EMA', ""]) & 
   (ydf['50_EMA', ""] < ydf['200_EMA', ""])
)

#Combine everything into one column to show bullish, bearish, divergence 
ydf['EMA_Alignment', ""] = 'divergence' 
ydf.loc[EMA_Bullish, ('EMA_Alignment', "")] = 'Bullish'
ydf.loc[EMA_Bearish, ('EMA_Alignment', "")] = 'Bearish'

ydf


Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,10_EMA,20_EMA,50_EMA,200_EMA,RSI,Upper_wick,Lower_wick,Upper_wick_percentage,Lower_wick_percentage,Wick_to_body_ratio,gap_size,gap_percentage,5EMA_touched,10EMA_touched,20EMA_touched,50EMA_touched,200EMA_touched,structure,confirmed_trend,EMA_Bullish,EMA_Alignment,5_day_return
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,,,,,,,,,,,,,,,,,,,,,,
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2015-01-02,0.482985,0.486584,0.475307,0.482985,113680000,0.000000,0.011277,Doji,0.000000,0.482985,0.482985,0.482985,0.482985,0.482985,NaN,0.003599,0.007678,31.915002,68.084998,inf,NaN,NaN,True,True,True,True,True,HL,0,False,divergence,0.009439
2015-01-05,0.474827,0.484425,0.472668,0.482985,197952000,0.008158,0.011757,Bearish,69.387800,0.480266,0.481502,0.482208,0.482665,0.482904,NaN,0.001440,0.002159,12.244977,18.367223,0.441176,-5.664483e-08,-0.000012,True,True,True,True,True,HL,0,False,Bearish,0.005053
2015-01-06,0.460432,0.476027,0.459952,0.475547,197764000,0.015116,0.016075,Bearish,94.029920,0.473655,0.477671,0.480134,0.481793,0.482681,NaN,0.000480,0.000480,2.985040,2.985040,0.063491,7.198833e-04,0.151609,True,False,False,False,False,HL,0,False,Bearish,-0.024492
2015-01-07,0.459232,0.467870,0.457792,0.463791,321808000,0.004559,0.010077,Bearish,45.237879,0.468847,0.474318,0.478144,0.480909,0.482447,NaN,0.004079,0.001440,40.476326,14.285795,1.210537,3.359050e-03,0.729544,False,False,False,False,False,LH,0,False,Bearish,-0.031348
2015-01-08,0.476507,0.479386,0.464270,0.464510,283780000,0.011997,0.015116,Bullish,79.364899,0.471400,0.474716,0.477988,0.480736,0.482388,NaN,0.002879,0.000240,19.047727,1.587374,0.260003,5.278483e-03,1.149415,True,True,True,False,True,HH,-1,False,Bearish,0.013091
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-26,192.529999,195.550003,191.220001,193.119995,179304100,0.589996,4.330002,Bearish,13.625776,198.058194,201.816238,205.671428,205.461208,188.614068,37.494618,2.430008,1.309998,56.120252,30.253972,6.339032,-2.620010e+00,-1.338516,False,False,False,False,False,LH,0,False,divergence,NaN
2026-06-29,194.970001,196.179993,189.800003,193.850006,148835700,1.119995,6.379990,Bullish,17.554811,197.028796,200.571468,204.652244,205.049789,188.677311,40.084358,1.209991,4.050003,18.965414,63.479775,4.696444,1.320007e+00,0.685611,False,False,False,False,False,HH,-1,False,divergence,NaN


In [179]:
EMA_Bullish

Date
2015-01-02    False
2015-01-05    False
2015-01-06    False
2015-01-07    False
2015-01-08    False
              ...  
2026-06-26    False
2026-06-29    False
2026-06-30    False
2026-07-01    False
2026-07-02    False
Length: 2891, dtype: bool

In [178]:
ydf['EMA_Alignment'].value_counts()

EMA_Alignment
divergence    2891
Name: count, dtype: int64

In [189]:
ydf['5_day_return'] = (ydf['Close', 'NVDA'] - ydf['Close', 'NVDA'].shift(-5)) / ydf['Close', 'NVDA']
ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,10_EMA,20_EMA,50_EMA,200_EMA,RSI,Upper_wick,Lower_wick,Upper_wick_percentage,Lower_wick_percentage,Wick_to_body_ratio,gap_size,gap_percentage,5EMA_touched,10EMA_touched,20EMA_touched,50EMA_touched,200EMA_touched,structure,confirmed_trend,EMA_Bullish,EMA_Alignment,5_day_return
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,,,,,,,,,,,,,,,,,,,,,,
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2015-01-02,0.482985,0.486584,0.475307,0.482985,113680000,0.000000,0.011277,Doji,0.000000,0.482985,0.482985,0.482985,0.482985,0.482985,NaN,0.003599,0.007678,31.915002,68.084998,inf,NaN,NaN,True,True,True,True,True,HL,0,False,divergence,0.009439
2015-01-05,0.474827,0.484425,0.472668,0.482985,197952000,0.008158,0.011757,Bearish,69.387800,0.480266,0.481502,0.482208,0.482665,0.482904,NaN,0.001440,0.002159,12.244977,18.367223,0.441176,-5.664483e-08,-0.000012,True,True,True,True,True,HL,0,False,Bearish,0.005053
2015-01-06,0.460432,0.476027,0.459952,0.475547,197764000,0.015116,0.016075,Bearish,94.029920,0.473655,0.477671,0.480134,0.481793,0.482681,NaN,0.000480,0.000480,2.985040,2.985040,0.063491,7.198833e-04,0.151609,True,False,False,False,False,HL,0,False,Bearish,-0.024492
2015-01-07,0.459232,0.467870,0.457792,0.463791,321808000,0.004559,0.010077,Bearish,45.237879,0.468847,0.474318,0.478144,0.480909,0.482447,NaN,0.004079,0.001440,40.476326,14.285795,1.210537,3.359050e-03,0.729544,False,False,False,False,False,LH,0,False,Bearish,-0.031348
2015-01-08,0.476507,0.479386,0.464270,0.464510,283780000,0.011997,0.015116,Bullish,79.364899,0.471400,0.474716,0.477988,0.480736,0.482388,NaN,0.002879,0.000240,19.047727,1.587374,0.260003,5.278483e-03,1.149415,True,True,True,False,True,HH,-1,False,Bearish,0.013091
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-26,192.529999,195.550003,191.220001,193.119995,179304100,0.589996,4.330002,Bearish,13.625776,198.058194,201.816238,205.671428,205.461208,188.614068,37.494618,2.430008,1.309998,56.120252,30.253972,6.339032,-2.620010e+00,-1.338516,False,False,False,False,False,LH,0,False,divergence,NaN
2026-06-29,194.970001,196.179993,189.800003,193.850006,148835700,1.119995,6.379990,Bullish,17.554811,197.028796,200.571468,204.652244,205.049789,188.677311,40.084358,1.209991,4.050003,18.965414,63.479775,4.696444,1.320007e+00,0.685611,False,False,False,False,False,HH,-1,False,divergence,NaN


In [ ]:
# Then we can use these variables to create a struncture for how we want the trend to look. 



# now we create trends 



# And now we can create a variable feature to identify these trends 


# No we will combine the results in column to see if we can find confirmed up or down trends

ydf['confirmed_trend'] = np.select(
    [uptrend_confirmed, downtrend_confirmed], [1, -1], default=0

In [20]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [32]:
pip install cpi

   ---------------------------------------- 0.0/18.1 MB ? eta -:--:--
   ----- ---------------------------------- 2.4/18.1 MB 18.5 MB/s eta 0:00:01
   ----------------- ---------------------- 8.1/18.1 MB 22.6 MB/s eta 0:00:01
   -------------------------------- ------- 14.9/18.1 MB 27.4 MB/s eta 0:00:01
   ---------------------------------------- 18.1/18.1 MB 26.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [47]:
pip install --upgrade cpi

Note: you may need to restart the kernel to use updated packages.


In [48]:
import cpi

In [49]:
df_cpi = cpi_series.to_dataframe()
df_cpi

,series_id,year,date,value,period_id,period_code,period_abbreviation,period_name,period_month,period_type
0,CUUR0000SA0,1997,1997-01-01,159.1,M01,M01,JAN,January,1,monthly
1,CUUR0000SA0,1997,1997-02-01,159.6,M02,M02,FEB,February,2,monthly
2,CUUR0000SA0,1997,1997-03-01,160.0,M03,M03,MAR,March,3,monthly
3,CUUR0000SA0,1997,1997-04-01,160.2,M04,M04,APR,April,4,monthly
4,CUUR0000SA0,1997,1997-05-01,160.1,M05,M05,MAY,May,5,monthly
...,...,...,...,...,...,...,...,...,...,...
1468,CUUR0000SA0,1996,1996-09-01,157.8,M09,M09,SEP,September,9,monthly
1469,CUUR0000SA0,1996,1996-10-01,158.3,M10,M10,OCT,October,10,monthly
1470,CUUR0000SA0,1996,1996-11-01,158.6,M11,M11,NOV,November,11,monthly
1471,CUUR0000SA0,1996,1996-12-01,158.6,M12,M12,DEC,December,12,monthly


In [22]:
ndf = yf.download("NVDA",
                  start="2015-01-01",
                  end="2026-04-30",
                  progress=False)
ndf

Price,Close,High,Low,Open,Volume
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA
Date,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000
2015-01-06,0.459896,0.475473,0.459416,0.474994,197764000
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000
...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400


In [50]:
ndf['5_EMA'] = talib.EMA(ndf['Close', 'NVDA'].to_numpy(), timeperiod=5)
ndf

Price,Close,High,Low,Open,Volume,5_EMA
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,
Date,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,NaN
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,NaN
2015-01-06,0.459896,0.475473,0.459416,0.474994,197764000,NaN
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,NaN
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,0.470249
...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,199.743683
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,202.504969


In [24]:
ydf['20)

SyntaxError: unterminated string literal (detected at line 1) (4034197693.py, line 1)

In [ ]:
this can be used to compare columns based on differnt days 

ydf.shift()

In [ ]:
ydf['Low']

In [ ]:
pip install nasdaq-data-link

In [ ]:
import nasdaqdatalink as ndl

In [ ]:
My_Nas_Key = "bp9V2Fn_MCJ8xEUxYS8p"

ndl.ApiConfig.api_key = My_Nas_Key

In [ ]:
ndf